In [2]:
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import pandas as pd
import sqlite3
import os
import html

# 1. Read notes_point layer from GeoPackage
file_name = "notes.gpkg"
gdf = gpd.read_file(file_name, layer="notes_point")
gdf = gdf.to_crs(epsg=4326)

# Image folder on your device
photo_folder = r"C:\Users\KANCH\OneDrive\UISA_26\DCIM"

# 2. Extract photo filenames from notes_attachments layer
photo_map = {}

try:
    conn = sqlite3.connect(file_name)
    cursor = conn.cursor()
    cursor.execute("SELECT note_uuid, media FROM notes_attachments;")
    rows = cursor.fetchall()
    conn.close()

    for uuid_val, media_path in rows:
        if media_path:
            clean_path = str(media_path).replace('\\', '/')
            photo_map[str(uuid_val).strip()] = clean_path

except Exception as e:
    print(f"Notice on attachments: {e}")

# 3. Initialize Interactive Map
mean_lat = gdf.geometry.y.mean()
mean_lon = gdf.geometry.x.mean()

m = folium.Map(
    location=[mean_lat, mean_lon],
    zoom_start=18,
    tiles=None,
    control_scale=True
)

# 4. Add Map Base Layers
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap &copy; CARTO",
    name="CartoDB Light Map",
    overlay=False
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri Satellite",
    name="Esri Satellite",
    overlay=False
).add_to(m)

marker_cluster = MarkerCluster(name="Waste Bins").add_to(m)

STUDENT_INFO = "Collected by: Roll No. hs25m105"

# Supported image extensions
image_extensions = {".jpg", ".jpeg", ".png", ".gif", ".webp", ".bmp"}

# 5. Populate Markers with Clear Popup Information
for idx, row in gdf.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x

    title = row.get('title') if pd.notna(row.get('title')) else f"Bin #{idx+1}"
    note = row.get('note') if pd.notna(row.get('note')) else "On road"
    timestamp = row.get('timestamp') if pd.notna(row.get('timestamp')) else "N/A"

    feat_uuid = str(row.get('uuid', '')).strip()
    rel_media_path = photo_map.get(feat_uuid, "")

    # Extract photo filename from the attachment path
    photo_filename = (
        rel_media_path.replace('\\', '/').split('/')[-1]
        if rel_media_path else ""
    )

    # Search for the actual image in the DCIM folder
    image_path = None

    if photo_filename and os.path.isdir(photo_folder):
        candidate = os.path.join(photo_folder, photo_filename)

        if os.path.isfile(candidate):
            image_path = candidate
        else:
            # Try case-insensitive filename matching
            try:
                for existing_file in os.listdir(photo_folder):
                    if existing_file.lower() == photo_filename.lower():
                        candidate = os.path.join(photo_folder, existing_file)
                        if os.path.isfile(candidate):
                            image_path = candidate
                            photo_filename = existing_file
                            break
            except OSError as e:
                print(f"Unable to search photo folder: {e}")

    # Display the actual image in the popup
    if image_path:
        image_url = "file:///" + image_path.replace("\\", "/").replace(" ", "%20")

        attachment_box = f"""
        <div style="margin-top: 8px; padding: 6px;
                    background-color: #f2f4f4;
                    border: 1px solid #d5dbdb; border-radius: 4px;
                    font-size: 11px;">
            <b> Photo Attachment:</b><br>
            <span style="color: #2e4053; word-break: break-all;">
                <code>{html.escape(photo_filename)}</code>
            </span>
            <br>
            <img src="{html.escape(image_url, quote=True)}"
                 alt="Waste bin photo"
                 style="display:block; width:100%; max-height:220px;
                        object-fit:contain; margin-top:6px;"
                 onerror="this.style.display='none';
                          this.nextElementSibling.style.display='block';">
            <span style="display:none; color:#922b21;">
                Image blocked by browser security settings.
            </span>
        </div>
        """

    elif photo_filename:
        attachment_box = f"""
        <div style="margin-top: 8px; padding: 6px;
                    background-color: #f2f4f4;
                    border: 1px solid #d5dbdb; border-radius: 4px;
                    font-size: 11px;">
            <b> Photo Attachment:</b><br>
            <code>{html.escape(photo_filename)}</code><br>
            <span style="color:#922b21;">
                Image not found in the DCIM folder.
            </span>
        </div>
        """

    else:
        attachment_box = """
        <div style="margin-top: 8px; padding: 6px;
                    background-color: #f2f4f4;
                    border: 1px solid #d5dbdb; border-radius: 4px;
                    font-size: 11px;">
            <b> Photo Attachment:</b><br>
            <span style="color:#922b21;">Not recorded</span>
        </div>
        """

    popup_content = f"""
    <div style="font-family: Arial, sans-serif; width: 220px;">
        <h4 style="margin: 0 0 6px 0; color: #1a5276;
                   border-bottom: 2px solid #2980b9; padding-bottom: 3px;">
            {html.escape(str(title))}
        </h4>
        <p style="margin: 3px 0; font-size: 12px;">
            <b>Student:</b> {html.escape(STUDENT_INFO)}
        </p>
        <p style="margin: 3px 0; font-size: 12px;">
            <b>Notes:</b> {html.escape(str(note))}
        </p>
        <p style="margin: 3px 0; font-size: 12px;">
            <b>Timestamp:</b> {html.escape(str(timestamp))}
        </p>
        {attachment_box}
    </div>
    """

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_content, max_width=260),
        tooltip=f"{title} ({STUDENT_INFO})",
        icon=folium.Icon(color='blue', icon='trash', prefix='fa')
    ).add_to(marker_cluster)

# 6. Add Layer Controls and save HTML file
folium.LayerControl(collapsed=False).add_to(m)

output_file = "waste_bin_map.html"
m.save(output_file)

print(f"Map successfully generated and saved to: {output_file}")
print(f"Photo folder used: {photo_folder}")


Map successfully generated and saved to: waste_bin_map.html
Photo folder used: C:\Users\KANCH\OneDrive\UISA_26\DCIM
